In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
# N = 1000
# P = 100
# dataset_folder = f"syn_data_N{N}_P{P}_num_seed_10"

N = 100
P = 100
dataset_folder = f'gtex_N{N}_P{P}__mode_partition'

results_dir = Path(os.environ["EPHEMERAL"]) / "pyspoc_runs/results/"
data_path = os.path.join(results_dir,'per_dataset',dataset_folder)
data_path = os.path.expanduser(data_path)

In [ ]:
results_dir

In [ ]:
configs_lst_all = os.listdir(data_path)

In [ ]:
configs_lst_all

In [ ]:
configs_lst = ['dist_distrib_stats_metricSweep',
 'manifold_stats',
 'pyspoc_internal_stats_fast']

In [ ]:
dfs_dct_config = {}

for config_nm in configs_lst:
    dfs_lst = []
    for filev in os.listdir(os.path.join(data_path,config_nm)):
        if filev.endswith('.parquet'):
            df = pd.read_parquet(os.path.join(data_path,config_nm,filev))
            dfs_lst.append(df)
    
    if len(dfs_lst)>0:        
        dfs_dct_config[config_nm] = pd.concat(dfs_lst)

In [ ]:
dataset_nan_thresh = 0.9
stat_nan_thresh = 0.9

for kv in ['pyspoc_internal_stats_fast']:
    
    concat_df = dfs_dct_config[kv]
    
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]

    print(kv)
    nan_per_dataset = concat_df.isna().sum(axis=1) / num_stats
    
    print('nans per dataset')
    print((nan_per_dataset>0).value_counts())
    print(nan_per_dataset.quantile(dataset_nan_thresh))
    
    nan_per_stat = concat_df.isna().sum(axis=0) / num_datasets

    print('nans per stat')    
    print((nan_per_stat>0).value_counts())
    print(nan_per_stat.quantile(stat_nan_thresh))    

    break

In [ ]:
# concat_df

In [ ]:
# chosen_configs

In [ ]:
def drop_nans(concat_df, stat_thresh_q=0.75, dataset_thresh_q=0.25):
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]
    
    old_nas = concat_df.isna().sum().sum()
    
    dct_outs = {}
    dct_outs["old NaNs"] = old_nas
    
    dct_outs["datasets tot"] = num_datasets
    dct_outs["stats tot"] = num_stats

    dct_outs["stats kept"] = num_stats
    dct_outs["datasets kept"] = num_datasets
    
    df = concat_df.copy()
    
    if old_nas > 0:
        
        stat_frac = df.isna().mean(axis=0)
        stat_thresh = stat_frac.quantile(stat_thresh_q)
        # stat_thresh = max(stat_thresh,0.1)
        df = df.loc[:, stat_frac <= stat_thresh]

        ds_frac = df.isna().mean(axis=1)
        dataset_thresh = ds_frac.quantile(dataset_thresh_q)

        df = df.loc[ds_frac <= dataset_thresh]

        dct_outs["stats kept"] = df.shape[1]
        dct_outs["datasets kept"] = df.shape[0]
        dct_outs["remaining NaNs"] = df.isna().sum().sum()
        dct_outs["stat_thresh"] = stat_thresh
        dct_outs["dataset_thresh"] = dataset_thresh
    
    print(dct_outs)
    return df,dct_outs

stat_q = 0.85
data_q = 0.25
clean = {kv: drop_nans(df,stat_thresh_q=stat_q,dataset_thresh_q=data_q) for kv, df in dfs_dct_config.items()}

In [ ]:
clean_dfs = {kv : out_tup[0] for kv,out_tup in clean.items()}
clean_stats = {kv : out_tup[1] for kv,out_tup in clean.items()}

concat_datamatrix = pd.concat(clean_dfs,axis=1,join="inner")

In [ ]:
concat_datamatrix.shape

In [ ]:
# for i in range(4):
#     print(concat_datamatrix[i::4].shape)

In [ ]:
all_datasets = set.union(*[set(df.index) for kv, df in dfs_dct_config.items()])
excluded_datasets = all_datasets - set(concat_datamatrix.index)
print(excluded_datasets)
print(len(all_datasets))
print(len(excluded_datasets))
print(set([v.split('__')[0] for v in excluded_datasets]))

In [ ]:
len([v.split('__')[0] for v in all_datasets if 'cpm_counts_scHPF' in v])

note- for synthetic data, the scHPF some of them are very sparse/overdispersed leading to Nans in some stats. worth investigating later

In [ ]:
# m.shape

In [ ]:
# clean_stats

In [ ]:
import json
from pathlib import Path

print(dataset_folder)
save_folder = os.path.join(os.path.expanduser(results_dir),"concat",dataset_folder)
os.makedirs(save_folder,exist_ok=True)

version=f'filtered_st{int(stat_q*100)}_dt{int(data_q*100)}'

m = concat_datamatrix.copy()
if isinstance(m.columns, pd.MultiIndex):
    m.columns = ["__".join(map(str, c)) for c in m.columns]

print(save_folder)
m.to_parquet(os.path.join(save_folder, f"datamatrix__{len(clean_stats)}configs__{version}.parquet"))

record = {
    "configs": clean_stats,
    "shape": list(m.shape),
    "created": pd.Timestamp.now().isoformat(),
    "excluded_datasets_type" : set([v.split('__')[0] for v in excluded_datasets]),
}

with open(os.path.join(save_folder,f"clean_stats__{version}.json"), "w") as f:
    json.dump(record, f, indent=2, default=lambda o : o.item() if hasattr(o, "item") else str(o))

In [ ]:
# clean_stats

In [ ]:
# concat_datamatrix

In [ ]:
# drop_nans(dfs_dct_config['pyspoc_internal_stats_fast'],stat_thresh_q=0.75)